In [1]:
import os
from pathlib import Path

import plotly.express as px
import torch
import torch.nn.functional as F
from dotenv import load_dotenv

from analysis.utils import load_autoencoder
from koopmann import aesthetics
from koopmann.llm import (
    LMHiddenStatesDataset,
    extract_hidden_states_from_hf,
    get_hf_llm,
    read_prompts,
)
from koopmann.utils import get_device
from koopmann.visualization import plot_eigenvalues

assert load_dotenv(Path.cwd().parent / ".env")

WEIGHTS_CACHE = os.getenv("WEIGHTS_CACHE")
assert WEIGHTS_CACHE is not None

HF_HOME = os.getenv("HF_HOME")

%load_ext autoreload
%autoreload 2

In [2]:
device = get_device()
model = "HuggingFaceTB/SmolLM-135M"
k = 10
dim = 800
seed = 22

autoencoder, metadata = load_autoencoder(
    WEIGHTS_CACHE,
    f"dim_{dim}_k_{k}_autoencoder_dummy_seed_{seed}",
)
autoencoder.to(device)
layer_i = 5
layer_j = 10

hf_model, hf_tokenizer = get_hf_llm(
    hf_name=model,
    cache_dir=HF_HOME,
    device=device,
)

_ = hf_model.eval()

In [3]:
# Make sure models are on the right device
autoencoder = autoencoder.to(device).eval()
hf_model = hf_model.to(device).eval()

In [4]:
# -----------------------------
# 1. Extract hidden states
# -----------------------------
prompt = "The cultural district on Saadiyat Island in"
hs, mask = extract_hidden_states_from_hf(
    model=hf_model,
    tokenizer=hf_tokenizer,
    prompts=[prompt],
    device=device,
)
# hs: (1, P, L+1, D), mask: (1, P)
last_tok = int(mask[0].sum().item() - 1)  # last non-pad token index

x_i = hs[0, last_tok, layer_i].float().to(device).unsqueeze(0)  # (1, D)
x_j = hs[0, last_tok, layer_j].float().to(device).unsqueeze(0)  # (1, D)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [5]:
x_i.shape

torch.Size([1, 576])

In [6]:
# -----------------------------
# 2. KAE prediction
# -----------------------------
with torch.no_grad():
    x_j_hat, _ = autoencoder(x_i)  # (1, D)
    x_j_hat = x_j_hat.squeeze(0)

# -----------------------------
# 3. Offline fidelity metrics
# -----------------------------
mse = torch.mean((x_j - x_j_hat) ** 2).item()
cos = F.cosine_similarity(x_j, x_j_hat).item()

# -----------------------------
# 4. Baseline next-token logits
# -----------------------------
tok = hf_tokenizer(prompt, return_tensors="pt").to(device)
out_base = hf_model(**tok, output_hidden_states=True, return_dict=True)
logits_base = out_base.logits[0, -1]  # logits for next token (final position)

baseline_token_id = torch.argmax(logits_base).item()
baseline_token = hf_tokenizer.decode([baseline_token_id])

# -----------------------------
# 5. Inject surrogate hidden state at layer_j
# -----------------------------
# Recall: hs[..., idx] comes from outputs.hidden_states[idx]
#   idx=0: embedding
#   idx=1: output of model.layers[0]
#   ...
#   idx=k: output of model.layers[k-1]
# So to match hs[..., layer_j], we hook model.layers[layer_j - 1].


def replace_layer_j(module, inp, out):
    if isinstance(out, tuple):
        hidden_states, *rest = out
        hidden_states = hidden_states.clone()
        hidden_states[:, last_tok, :] = x_j_hat[0]
        return (hidden_states, *rest)
    else:
        hs = out.clone()
        hs[:, last_tok, :] = x_j_hat[0]
        return hs


hook_layer_index = layer_j - 1  # align with hidden_states indexing
handle = hf_model.model.layers[hook_layer_index].register_forward_hook(replace_layer_j)

# Forward with injection
out_inj = hf_model(**tok, output_hidden_states=True, return_dict=True)
logits_inj = out_inj.logits[0, -1]

kae_token_id = torch.argmax(logits_inj).item()
kae_token = hf_tokenizer.decode([kae_token_id])

handle.remove()

# -----------------------------
# 6. Logit comparison metrics
# -----------------------------
logit_mse = torch.mean((logits_base - logits_inj) ** 2).item()
logit_cos = F.cosine_similarity(logits_base, logits_inj, dim=0).item()

print(f"Prompt: {prompt}\n")

print("Offline Fidelity Test")
print("---------------------")
print(f"MSE:         {mse:.6f}")
print(f"Cosine Sim:  {cos:.6f}")

print("\nOnline Token Fidelity Test")
print("--------------------------")
print(f"Baseline token: {baseline_token!r}")
print(f"KAE token:      {kae_token!r}")
print(f"Token IDs:      {baseline_token_id} vs {kae_token_id}")
print(f"Logit MSE:      {logit_mse:.6f}")
print(f"Logit Cosine:   {logit_cos:.6f}")


Prompt: The cultural district on Saadiyat Island in

Offline Fidelity Test
---------------------
MSE:         2.172500
Cosine Sim:  0.975704

Online Token Fidelity Test
--------------------------
Baseline token: ' the'
KAE token:      ' the'
Token IDs:      260 vs 260
Logit MSE:      1.125000
Logit Cosine:   1.007812


In [ ]:
import torch
import torch.nn.functional as F

# ---------------------------------------------
# 0. Extract the Koopman matrix
# ---------------------------------------------
K = autoencoder.components.koopman_matrix.components.linear.weight.detach().cpu()
eigvals, eigvecs = torch.linalg.eig(K)

print("Top 10 eigenvalues (original):")
print(eigvals[:5], "\n")

# ---------------------------------------------
# 1. Choose eigenmode and scale
# ---------------------------------------------
k_idx = [0]  # which eigenmode to poke
scale = 1.001  # <1 suppress, >1 amplify

print(f"Modifying eigenvalue {k_idx}:")
print(f"  original λ_k = {eigvals[k_idx]}")
print(f"  new λ_k     = {eigvals[k_idx] * scale}\n")

# ---------------------------------------------
# 2. Build modified Koopman matrix K_mod
# ---------------------------------------------
eigvals_mod = eigvals.clone()
eigvals_mod[k_idx] *= scale

V = eigvecs
V_inv = torch.linalg.inv(V)
Lambda_mod = torch.diag(eigvals_mod)

K_mod_c = V @ Lambda_mod @ V_inv
K_mod = K_mod_c.real.to(K.dtype)  # strip imag part

print(f"‖K_mod - K‖_F = {torch.norm(K_mod - K).item():.6f}\n")

# ---------------------------------------------
# 3. Pick prompt and extract x_i, x_j
# ---------------------------------------------
prompt = "Walking along the Corniche is a"
hs, mask = extract_hidden_states_from_hf(
    model=hf_model, tokenizer=hf_tokenizer, prompts=[prompt], device=device
)

last_tok = int(mask[0].sum() - 1)

layer_i = 5
layer_j = 10

x_i = hs[0, last_tok, layer_i].float().to(device).unsqueeze(0)
x_j_true = hs[0, last_tok, layer_j].float().to(device).unsqueeze(0)

# ---------------------------------------------
# 4. Standard KAE prediction (baseline)
# ---------------------------------------------
with torch.no_grad():
    z = autoencoder.encode(x_i)
    z = autoencoder.koopman_forward(z)
    x_j_hat = autoencoder.decode(z)

print("Baseline KAE prediction vs true:")
print(f"  cos(x_j_true, x_j_hat) = {F.cosine_similarity(x_j_true, x_j_hat).item():.6f}")
print(f"  mse(x_j_true, x_j_hat) = {torch.mean((x_j_true - x_j_hat)**2).item():.6f}\n")

# ---------------------------------------------
# 5. Eigen-edited K prediction
# ---------------------------------------------
K_mod_dev = K_mod.to(device)

with torch.no_grad():
    phi = autoencoder.encode(x_i)
    z_mod = phi @ torch.linalg.matrix_power(K_mod_dev.T, autoencoder.k_steps)
    x_j_mod = autoencoder.decode(z_mod)

print("Modified KAE prediction (after eigen edit):")
print(f"  cos(x_j_hat,  x_j_mod) = {F.cosine_similarity(x_j_hat, x_j_mod).item():.6f}")
print(f"  cos(x_j_true, x_j_mod) = {F.cosine_similarity(x_j_true, x_j_mod).item():.6f}")
print(f"  mse(x_j_true, x_j_mod) = {torch.mean((x_j_true - x_j_mod)**2).item():.6f}\n")

# ---------------------------------------------
# 6. Compare next token (baseline LM vs modified LM)
# ---------------------------------------------
# --- Baseline token ---
tok = hf_tokenizer(prompt, return_tensors="pt").to(device)
out_base = hf_model(**tok, output_hidden_states=True, return_dict=True)
logits_base = out_base.logits[0, -1]
baseline_id = torch.argmax(logits_base).item()
baseline_tok = hf_tokenizer.decode([baseline_id])


# --- Inject x_j_mod at layer_j ---
def replace_layer_j(module, inp, out):
    if isinstance(out, tuple):
        h, *rest = out
        h = h.clone()
        h[:, last_tok, :] = x_j_mod[0]
        return (h, *rest)
    else:
        h = out.clone()
        h[:, last_tok, :] = x_j_mod[0]
        return h


hook_layer_index = layer_j - 1
handle = hf_model.model.layers[hook_layer_index].register_forward_hook(replace_layer_j)

out_mod = hf_model(**tok, output_hidden_states=True, return_dict=True)
logits_mod = out_mod.logits[0, -1]

handle.remove()

mod_id = torch.argmax(logits_mod).item()
mod_tok = hf_tokenizer.decode([mod_id])

print("Next-token behavior:")
print(f"  Baseline token: {baseline_tok!r} ({baseline_id})")
print(f"  Modified token: {mod_tok!r}  ({mod_id})")
print(
    f"  Logit cosine: {F.cosine_similarity(logits_base, logits_mod, dim=0).item():.6f}"
)
print(f"  Logit MSE:    {torch.mean((logits_base - logits_mod)**2).item():.6f}")


# ---------------------------------------------
# 7. Multiple-step generation with the same nudge
# ---------------------------------------------
def generate_baseline(prompt, n_steps=20):
    text = prompt
    for _ in range(n_steps):
        tok = hf_tokenizer(text, return_tensors="pt").to(device)
        with torch.no_grad():
            out = hf_model(**tok, return_dict=True)
            logits = out.logits[0, -1]
            next_id = int(torch.argmax(logits))
        text += hf_tokenizer.decode([next_id])
    return text


def generate_with_eigen_nudge(prompt, n_steps=20):
    text = prompt
    for _ in range(n_steps):
        # 1) Run model once to get hidden states at layer_i
        tok = hf_tokenizer(text, return_tensors="pt").to(device)
        with torch.no_grad():
            out = hf_model(**tok, output_hidden_states=True, return_dict=True)
        attn = tok["attention_mask"]
        last_tok_step = int(attn[0].sum().item() - 1)

        # hidden_states: tuple len L+1, each [B, P, D]
        hs_step = out.hidden_states
        x_i_step = hs_step[layer_i][:, last_tok_step, :].float().to(device)  # [1, D]

        # 2) Encode -> Koopman with K_mod -> Decode
        with torch.no_grad():
            phi = autoencoder.encode(x_i_step)
            z_mod = phi @ torch.linalg.matrix_power(K_mod_dev.T, 1)
            x_j_mod_step = autoencoder.decode(z_mod)  # [1, D]

        # 3) Second forward pass with hook injecting x_j_mod_step at layer_j
        def replace_layer_j_step(module, inp, out):
            if isinstance(out, tuple):
                h, *rest = out
                h = h.clone()
                h[:, last_tok_step, :] = x_j_mod_step[0]
                return (h, *rest)
            else:
                h = out.clone()
                h[:, last_tok_step, :] = x_j_mod_step[0]
                return h

        hook_layer_index = layer_j - 1
        handle = hf_model.model.layers[hook_layer_index].register_forward_hook(
            replace_layer_j_step
        )

        with torch.no_grad():
            out_mod = hf_model(**tok, return_dict=True)
            logits_mod_step = out_mod.logits[0, -1]
            next_id = int(torch.argmax(logits_mod_step))

        handle.remove()

        text += hf_tokenizer.decode([next_id])

    return text


n_steps = 20

baseline_gen = generate_baseline(prompt, n_steps=n_steps)
nudged_gen = generate_with_eigen_nudge(prompt, n_steps=n_steps)

print("\n=== Baseline generation ===")
print(baseline_gen)
print("\n=== Nudged (eigen-edited K) generation ===")
print(nudged_gen)


Top 10 eigenvalues (original):
tensor([0.9775+0.0000j, 1.0045+0.0000j, 1.0321+0.0157j, 1.0321-0.0157j,
        1.0306+0.0120j]) 

Modifying eigenvalue [0]:
  original λ_k = tensor([0.9775+0.j])
  new λ_k     = tensor([0.9785+0.j])

‖K_mod - K‖_F = 0.134292

Baseline KAE prediction vs true:
  cos(x_j_true, x_j_hat) = 0.928048
  mse(x_j_true, x_j_hat) = 7.280163

Modified KAE prediction (after eigen edit):
  cos(x_j_hat,  x_j_mod) = 0.999987
  cos(x_j_true, x_j_mod) = 0.927863
  mse(x_j_true, x_j_mod) = 7.336231

Next-token behavior:
  Baseline token: ' popular' (2378)
  Modified token: ' a'  (253)
  Logit cosine: 0.486328
  Logit MSE:    10.437500

=== Baseline generation ===
Walking along the Corniche is a popular way to get to know the city. The Corniche is a series of narrow, winding streets

=== Nudged (eigen-edited K) generation ===
Walking along the Corniche is a the- a-- of-...,, a,,,,,,,
